# Aadhaar OCR Test

Extracts Aadhaar number, DOB, Gender, and Address from an Aadhaar card image using EasyOCR.

In [4]:
import cv2
import easyocr
import re
import os

reader = easyocr.Reader(['en', 'hi'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [5]:
def extract_aadhaar(image_path):
    if not os.path.exists(image_path):
        print(f"Error: Image file '{image_path}' not found!")
        return None

    print(f"Reading image: {image_path}")
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    print("Running text extraction...")
    results = reader.readtext(gray, detail=1)

    texts = [r[1] for r in results]
    full_text = " ".join(texts)

    print("\n--- Raw Text Extracted ---")
    for i, (bbox, text, conf) in enumerate(results):
        y = int(bbox[0][1])
        print(f"  [{i:>2}] [y={y:>4}] {text}")
    print("--------------------------\n")

    aadhaar_pattern = r'\b\d{4}\s?\d{4}\s?\d{4}\b'
    dob_pattern = r'(?:DOB|Date of Birth|Year of Birth)[:\s]*(\d{2}/\d{2}/\d{4}|\d{4})'
    gender_pattern = r'\b(Male|Female|MALE|FEMALE|TRANSGENDER)\b'
    name_pattern = r'Government of India[^A-Za-z]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'
    father_pattern = r'Father[:\s]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'

    aadhaar_matches = re.findall(aadhaar_pattern, full_text)
    aadhaar_number = aadhaar_matches[0].replace(' ', '') if aadhaar_matches else None

    # Card type: long cards have "Your Aadhaar No." label section
    aadhaar_label_idx = None
    for i, (bbox, text, _) in enumerate(results):
        if re.search(r'Aadhaar No', text):
            aadhaar_label_idx = i
            break

    card_type = "long" if aadhaar_label_idx is not None else "short"

    dob = re.search(dob_pattern, full_text, re.IGNORECASE)
    gender = re.search(gender_pattern, full_text, re.IGNORECASE)
    name = re.search(name_pattern, full_text)
    father = re.search(father_pattern, full_text, re.IGNORECASE)

    address = "Not Applicable"
    if aadhaar_label_idx is not None and aadhaar_label_idx > 4:
        addr_texts = []
        for i in range(4, aadhaar_label_idx):
            text = results[i][1].strip()
            normalized = text.translate(str.maketrans('०१२३४५६७८९', '0123456789'))
            if not normalized.isascii():
                continue
            if re.search(r'[A-Z]{2}\d{6,}', normalized):
                continue
            addr_texts.append(normalized)
        address = ", ".join(addr_texts) if addr_texts else "Not Found"

    result = {
        "status": "success",
        "card_type": card_type,
        "aadhaar_found": aadhaar_number is not None,
        "data": {
            "aadhaar_number": aadhaar_number,
            "name": name.group(1) if name else "Not Found",
            "father_name": father.group(1) if father else "Not Found",
            "dob": dob.group(1) if dob else "Not Found",
            "gender": gender.group(0).upper() if gender else "Not Found",
            "address": address
        }
    }

    print("Parsed JSON Result:")
    return result

In [6]:
image_path = "ss.jpeg"
extract_aadhaar(image_path)

Reading image: ss.jpeg
Running text extraction...

--- Raw Text Extracted ---
  [ 0] [y= 170] To
  [ 1] [y= 200] तेजे॰
  [ 2] [y= 200] बमे॰
  [ 3] [y= 232] Telen Barman
  [ 4] [y= 219] डै
  [ 5] [y= 264] PATHAR COLONY
  [ 6] [y= 300] GHOKLAJOTE
  [ 7] [y= 302] MATIGARA
  [ 8] [y= 330] Gaurcharan
  [ 9] [y= 361] Matigara
  [10] [y= 393] Matigara Darjeeling
  [11] [y= 427] West Bengal 7३4010
  [12] [y= 401] डूै
  [13] [y= 511] MA249552486F
  [14] [y= 694] आपका
  [15] [y= 696] आधार
  [16] [y= 698] क्रमांक / Your Aadhaar No
  [17] [y= 717] -
  [18] [y= 761] 2152
  [19] [y= 761] 4563
  [20] [y= 765] 1287
  [21] [y= 851] आधार
  [22] [y= 860] आम
  [23] [y= 853] आदमी
  [24] [y= 865] का
  [25] [y= 853] अधिकार
  [26] [y= 979] भारत
  [27] [y= 982] सरकार
  [28] [y=1016] Government of India
  [29] [y=1062] तेजेन
  [30] [y=1062] बर्मन
  [31] [y=1095] Tejen
  [32] [y=1099] Barman
  [33] [y=1130] पिता
  [34] [y=1130] भालो
  [35] [y=1130] बर्मन
  [36] [y=1166] Father
  [37] [y=1166] Bhalo Barman
  [38]

{'status': 'success',
 'card_type': 'long',
 'aadhaar_found': True,
 'data': {'aadhaar_number': '215245631287',
  'name': 'Tejen Barman',
  'father_name': 'Bhalo Barman',
  'dob': '01/01/1965',
  'gender': 'MALE',
  'address': 'PATHAR COLONY, GHOKLAJOTE, MATIGARA, Gaurcharan, Matigara, Matigara Darjeeling, West Bengal 734010'}}